# Log-normalizing Pareto observations and estimating the tail index

A Pareto distribution with minimum $x_{\min}>0$ and tail index $α>0$ has survival function

$$\bar F(x)=\Pr(X>x)=\left(\frac{x_{\min}}{x}\right)^α,\quad x\ge x_{\min}.$$

Its density is $f(x)=α x_{\min}^α/x^{α+1}$.

**Key observation.** If $X$ has Pareto parameters $(x_{\min},α)$, then

$$Y:=\log\left(\frac{X}{x_{\min}}\right)\sim\mathrm{Exp}(α).$$

Indeed, $\Pr(Y>y)=\Pr(X>x_{\min}e^y)=e^{-αy}$ for $y\ge0$. The log transformation converts the Pareto power-law tail into an exponential tail.

**Consequence: Hill / maximum-likelihood estimation.** Since $\mathbb E[Y]=1/α$,

$$\hat α_{\mathrm{Hill}}=\frac{1}{\bar Y}=\frac{n}{\sum_{i=1}^n\log(X_i/x_{\min})}.$$

Some references use a different symbol for the tail exponent. Extreme-value references may instead report its reciprocal, so the convention must be checked.

The asymptotic standard error is approximately $\hat α/\sqrt n$, as for the exponential-rate MLE. The cells below examine these relationships numerically.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(seed=42)

# True parameters
alpha_true = 2.5      # tail index (could be called k)
x_min = 1.0           # location / lower bound
n = 5000              # sample size

# Pareto via inverse CDF: U ~ U(0,1) ⇒ X = x_min · (1 - U)^(-1/α)
u = rng.uniform(size=n)
X = x_min * (1 - u) ** (-1.0 / alpha_true)

print(f"Min, max of X:        {X.min():.3f}, {X.max():.3f}")
print(f"Median of X:          {np.median(X):.3f}")
print(f"Mean of X (theory):   {alpha_true / (alpha_true - 1) * x_min:.3f}   (α/(α−1)·x_min)")
print(f"Mean of X (sample):   {X.mean():.3f}")
print()
print(f"Variance is finite iff α > 2.   Here α = {alpha_true}, so yes.")
print(f"Sample variance:      {X.var():.3f}")

## 1. The Pareto tail on the original scale

On a linear scale, $X$ is strongly skewed and rare large observations dominate the sample. A log-log plot of empirical survival makes the power-law tail visible.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Histogram on linear scale.
axes[0].hist(X[X < np.quantile(X, 0.99)], bins=60, color="steelblue", alpha=0.8)
axes[0].set_xlabel("X")
axes[0].set_ylabel("count")
axes[0].set_title("Histogram of X on a linear scale (below the 99th percentile)")

# Empirical survival on log-log scale.
X_sorted = np.sort(X)
surv = 1 - np.arange(1, n + 1) / n
# Drop the rightmost point (survival = 0, log is -∞).
ax = axes[1]
ax.loglog(X_sorted[:-1], surv[:-1], "o", ms=2, alpha=0.5, label="empirical $\\bar F$")
# Theoretical line: survival ∝ x^(-α).
xx = np.linspace(x_min, X_sorted[-2], 100)
ax.loglog(xx, (x_min / xx) ** alpha_true, "r--", lw=2,
           label=f"$(x_{{\\min}}/x)^{{{alpha_true}}}$")
ax.set_xlabel("x (log scale)")
ax.set_ylabel("$\\bar F(x)$ (log scale)")
ax.set_title("Empirical survival function (log-log)")
ax.legend()
plt.tight_layout(); plt.show()

A straight line with slope $-α$ on log-log axes characterizes a power-law tail.

## 2. Log transformation: Pareto to exponential

Apply $Y=\log(X/x_{\min})$. Its theoretical law is $\mathrm{Exp}(α)$, with density $αe^{-αy}$ on $[0,\infty)$.


In [ ]:
Y = np.log(X / x_min)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Histogram of Y, with the Exp(α_true) PDF overlaid.
axes[0].hist(Y, bins=60, density=True, color="seagreen", alpha=0.7,
             label="histogram of Y")
yy = np.linspace(0, Y.max(), 200)
axes[0].plot(yy, alpha_true * np.exp(-alpha_true * yy), "r-", lw=2,
             label=f"Exp($\\alpha$={alpha_true}) PDF")
axes[0].set_xlabel("Y = log(X / x_min)")
axes[0].set_ylabel("density")
axes[0].set_title("Y after the log transformation (expected law: Exp(α))")
axes[0].legend()

# Q-Q plot of Y vs theoretical Exp(α_true).
(osm, osr), (slope, intercept, r) = stats.probplot(
    Y, dist=stats.expon, sparams=(0, 1.0 / alpha_true), plot=axes[1])
axes[1].set_title(f"Q-Q plot: Y versus Exp(α={alpha_true})")
axes[1].set_xlabel("theoretical quantiles")
axes[1].set_ylabel("empirical quantiles")
plt.tight_layout(); plt.show()

print(f"Q–Q regression slope:     {slope:.4f}   (expected ≈ 1)")
print(f"Q–Q regression intercept: {intercept:.4f}  (expected ≈ 0)")
print(f"Q–Q R²:                   {r**2:.5f}   (expected ≈ 1)")

## 3. Estimating the tail index: Hill / MLE

Since $Y\sim\mathrm{Exp}(α)$ has mean $1/α$, the maximum-likelihood estimator is

$$\hat α=\frac{1}{\bar Y}=\frac{n}{\sum_i\log(X_i/x_{\min})}.$$

This is the known-threshold Pareto version of the Hill estimator. Its asymptotic distribution is

$$\sqrt n(\hat α-α)\stackrel d\longrightarrow\mathcal N(0,α^2),$$

giving the estimated standard error $\hat α/\sqrt n$.


In [ ]:
alpha_hat = 1.0 / Y.mean()
se_hat = alpha_hat / np.sqrt(n)
ci95 = (alpha_hat - 1.96 * se_hat, alpha_hat + 1.96 * se_hat)

print(f"True α:                 {alpha_true}")
print(f"Hill / MLE estimate:    α̂ = {alpha_hat:.4f}")
print(f"Standard error:         SE(α̂) ≈ {se_hat:.4f}")
print(f"95% confidence interval: [{ci95[0]:.4f}, {ci95[1]:.4f}]")
print(f"Inside CI:              {ci95[0] <= alpha_true <= ci95[1]}")

## 4. Convergence with sample size

Examine how $\hat α$ approaches $α$ as $n$ increases, and compare the empirical variability with the asymptotic variance $α^2/n$.


In [ ]:
ns = np.array([50, 100, 200, 500, 1000, 2000, 5000, 10000])
n_replicas = 400

alpha_hats = np.empty((len(ns), n_replicas))
for i, ni in enumerate(ns):
    u_block = rng.uniform(size=(n_replicas, ni))
    X_block = x_min * (1 - u_block) ** (-1.0 / alpha_true)
    Y_block = np.log(X_block / x_min)
    alpha_hats[i] = 1.0 / Y_block.mean(axis=1)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.axhline(alpha_true, color="red", lw=2, label=f"true α = {alpha_true}")
ax.errorbar(ns, alpha_hats.mean(axis=1),
            yerr=alpha_hats.std(axis=1),
            fmt="o", capsize=4, color="steelblue",
            label="mean α̂ ± std (400 replications)")
# Theoretical asymptotic SE: α/√n.
se_theory = alpha_true / np.sqrt(ns)
ax.fill_between(ns, alpha_true - 1.96 * se_theory, alpha_true + 1.96 * se_theory,
                alpha=0.15, color="red", label="95% band (α/√n)")
ax.set_xscale("log")
ax.set_xlabel("sample size n")
ax.set_ylabel("estimate α̂")
ax.set_title("Convergence of the Hill estimate to the true α")
ax.legend()
plt.tight_layout(); plt.show()

print("   n     empirical SE     theoretical α/√n")
for ni, row in zip(ns, alpha_hats):
    print(f"  {ni:5d}     {row.std():.4f}            {alpha_true/np.sqrt(ni):.4f}")

## 5. Choosing the threshold

For real data, $x_{\min}$ is usually unknown. Tail-index estimation uses observations above a selected threshold. With increasing order statistics, the threshold-based Hill estimate is

$$\hat α_k=\frac{k}{\sum_{i=1}^k\log(X_{(n-i+1)}/X_{(n-k)})}.$$

Here $X_{(n-k)}$ is the empirical threshold. A Hill plot displays $\hat α_k$ against the number $k$ of upper-tail observations.


In [ ]:
X_sorted = np.sort(X)  # ascending
ks = np.arange(20, n)
alpha_k = np.empty_like(ks, dtype=float)
for j, k in enumerate(ks):
    # Top-k order statistics: X_sorted[-k:]; threshold X_sorted[-k-1].
    top = X_sorted[-k:]
    threshold = X_sorted[-k - 1]
    alpha_k[j] = k / np.sum(np.log(top / threshold))

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.axhline(alpha_true, color="red", lw=2, label=f"true α = {alpha_true}")
ax.plot(ks, alpha_k, color="steelblue", lw=1)
ax.fill_between(ks,
                alpha_k - 1.96 * alpha_k / np.sqrt(ks),
                alpha_k + 1.96 * alpha_k / np.sqrt(ks),
                alpha=0.2, color="steelblue", label="95% CI")
ax.set_xlabel("k (number of upper-tail order statistics)")
ax.set_ylabel("$\\hat\\alpha_k$")
ax.set_title("Hill plot: inspect the stability of the estimated tail index")
ax.legend()
plt.tight_layout(); plt.show()

## 6. How different is this from a Gaussian?

Fit $\mathcal N(\hat μ,\hat σ^2)$ to $X$ and compare several diagnostics.

**Moment existence.** A Pareto $k$th moment is finite exactly when $α>k$; Gaussian moments are finite at every positive order. For $α=2.5$:

| Quantity | Pareto, $α=2.5$ | Gaussian |
| --- | --- | --- |
| Mean | Finite | Finite |
| Variance | Finite | Finite |
| Third absolute moment / skewness | No finite third moment | Finite; skewness 0 |
| Fourth moment / excess kurtosis | No finite fourth moment | Finite; excess kurtosis 0 |

Empirical skewness and excess kurtosis can be large and unstable. This is a structural difference from a Gaussian law even when the variance exists.


In [ ]:
# Moments comparison
mu_g, sigma_g = X.mean(), X.std()

print(f"Pareto α = {alpha_true}  —  moment existence:")
print(f"  E[X]      finite when α > 1:  {alpha_true > 1}")
print(f"  Var(X)    finite when α > 2:  {alpha_true > 2}")
print(f"  Skewness  finite when α > 3:  {alpha_true > 3}  ← not finite in this example")
print(f"  Kurtosis  finite when α > 4:  {alpha_true > 4}  ← not finite in this example")
print()
print(f"Empirical values (may vary substantially between runs):")
print(f"  mean      X.mean() = {X.mean():.4f}")
print(f"  std       X.std()  = {X.std():.4f}")
print(f"  skewness            = {stats.skew(X):.4f}      (Gauss: 0)")
print(f"  excess kurtosis     = {stats.kurtosis(X):.4f}    (Gauss: 0)")

In [ ]:
# Q–Q plot against best-fit Gaussian — should curve sharply in the right tail.
fig, ax = plt.subplots(figsize=(7.5, 5))
stats.probplot(X, dist=stats.norm, sparams=(mu_g, sigma_g), plot=ax)
ax.set_title(f"Q-Q plot: X versus $\\mathcal{{N}}({mu_g:.2f}, {sigma_g:.2f}^2)$")
ax.set_xlabel("Gaussian quantiles")
ax.set_ylabel("sample quantiles")
ax.get_lines()[1].set_linewidth(2)
plt.tight_layout(); plt.show()

In [ ]:
# Tail probabilities: Pareto vs Gaussian fit, vs empirical.
fig, ax = plt.subplots(figsize=(9, 5))
xx = np.linspace(max(mu_g, x_min) * 1.01, np.quantile(X, 0.9995), 400)
surv_pareto = (x_min / xx) ** alpha_true
surv_gauss = stats.norm.sf(xx, loc=mu_g, scale=sigma_g)

X_sorted = np.sort(X)
emp_s = 1 - np.arange(1, n + 1) / n

ax.semilogy(xx, surv_pareto, "r-", lw=2, label=f"Pareto ($\\alpha={alpha_true}$)")
ax.semilogy(xx, surv_gauss, "b--", lw=2, label=f"$\\mathcal{{N}}({mu_g:.2f}, {sigma_g:.2f}^2)$")
ax.semilogy(X_sorted[:-1], emp_s[:-1], "k.", ms=2, alpha=0.3, label="empirical")
ax.set_xlabel("x"); ax.set_ylabel("$\\bar F(x)$  (log)")
ax.set_title("Survival: Pareto versus a Gaussian with the same $\\hat\\mu, \\hat\\sigma$")
ax.set_ylim(1e-12, 1)
ax.legend()
plt.tight_layout(); plt.show()

print("Tail-probability ratios at different thresholds:")
for k_sigma in [1, 2, 3, 4, 5, 6]:
    x_k = mu_g + k_sigma * sigma_g
    if x_k > x_min:
        sp = (x_min / x_k) ** alpha_true
        sg = stats.norm.sf(x_k, loc=mu_g, scale=sigma_g)
        ratio = sp / sg if sg > 0 else float('inf')
        print(f"  P(X > μ + {k_sigma}σ):  Pareto = {sp:.2e},  Gaussian = {sg:.2e},  ratio = {ratio:.1e}×")

In [ ]:
# Statistical tests for normality.
from scipy.stats import shapiro, anderson, kstest

X_sub = X[:5000] if n > 5000 else X
stat_sw, p_sw = shapiro(X_sub)
print(f"Shapiro–Wilk on X:     stat = {stat_sw:.6f},  p-value = {p_sw:.3e}")
print(f"  → {'reject' if p_sw < 0.05 else 'do not reject'} the Gaussian null at the 5% level")

a_d = anderson(X, dist="norm")
print(f"\nAnderson–Darling on X: stat = {a_d.statistic:.4f}")
print(f"  Critical values at significance levels {tuple(a_d.significance_level)}:")
print(f"  {a_d.critical_values}")
print(f"  → reject at all reported levels {'(stat >> max crit value)' if a_d.statistic > a_d.critical_values.max() else ''}")

ks_stat, ks_p = kstest(X, lambda v: stats.norm.cdf(v, loc=mu_g, scale=sigma_g))
print(f"\nKolmogorov–Smirnov on X vs N(μ̂,σ̂²): stat = {ks_stat:.4f},  p = {ks_p:.3e}")

# For reference, the same test on Y (which is exponential, also not Gaussian).
Y_sub = Y[:5000] if n > 5000 else Y
stat_y, p_y = shapiro(Y_sub)
print(f"\nFor reference, Y = log(X/x_min) is exponential and also non-Gaussian:")
print(f"  Shapiro–Wilk on Y:   stat = {stat_y:.6f},  p-value = {p_y:.3e}")

In [ ]:
# How does the sample maximum scale?
n_max = n
gauss_max_typical = np.sqrt(2 * np.log(n_max))
pareto_max_typical = n_max ** (1.0 / alpha_true)

print(f"Typical maximum scale for sample size n = {n}:")
print(f"  Gaussian N(0,1):   max ≈ √(2 log n) = {gauss_max_typical:.2f}  (in units of σ)")
print(f"  Pareto α = {alpha_true}:    max ≈ n^(1/α)  = {pareto_max_typical:.1f}  (in units of x_min)")
print()
print(f"Empirical maximum:                X.max()   = {X.max():.2f}")
print(f"Standard deviations above the sample mean:             (X.max() − μ̂) / σ̂ = {(X.max() - mu_g) / sigma_g:.2f}σ")
tail_prob = stats.norm.sf((X.max() - mu_g) / sigma_g)
print(f"Gaussian tail probability of this deviation: {tail_prob:.3e}")
print(f"  ← under the fitted Gaussian model, such an exceedance would occur about")
print(f"     once per {1/tail_prob:.1e} observations. It occurs here in a sample of {n}.")

**Diagnostics of the departure from a Gaussian model:**

| Diagnostic | Gaussian reference | Pareto, $α=2.5$ |
| --- | --- | --- |
| Skewness | 0 | No finite third moment; unstable empirical values |
| Excess kurtosis | 0 | No finite fourth moment |
| Upper-tail Q-Q plot | Approximately straight | Strong upward deviation |
| Probability above $μ+5σ$ | About $3\cdot10^{-7}$ | Can be much larger |
| Normality-test p-value | Random under a correctly specified null | Often very small in this experiment |
| Typical maximum growth | Order $\sqrt{\log n}$ | Order $n^{1/α}$ |

A Gaussian fitted to the same sample mean and standard deviation can substantially underestimate remote-tail probabilities. The Q-Q and survival plots display this mismatch.

The transformation $Y=\log(X/x_{\min})$ removes the Pareto power-law tail, but the resulting exponential law is still asymmetric and non-Gaussian. Normality-test results on $Y$ should be interpreted accordingly. Tests that fit parameters to the same sample may require adjusted null distributions.


## Conclusions

1. The log transformation $Y=\log(X/x_{\min})$ converts a Pareto law into $\mathrm{Exp}(α)$, replacing power-law tail decay with exponential decay.
2. The known-threshold Pareto MLE is $\hat α=n/\sum_i\log(X_i/x_{\min})$.
3. The experiment is consistent with the asymptotic distribution $\sqrt n(\hat α-α)\to\mathcal N(0,α^2)$.
4. A Hill plot helps examine stability when choosing an empirical tail threshold.

A slope fitted to log-log empirical survival also provides a tail-index estimate, but it is not generally equivalent to the likelihood estimator.
